In [ ]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score



from gensim.models import Word2Vec
from collections import Counter
import joblib

Load Dataset yang Sudah di Preprocessing

In [2]:
df = pd.read_csv('../data/dataset_processing.csv')
df.head()

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source,category,cleaned_text
0,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara","Senin, 12 Agu 2024 21:58 WIB",https://news.detik.com/berita/d-7486691/gempa-...,"Gempa bumi berkekuatan magnitudo (M) 5,3 mengg...",pulau doi,gempa,NaN,NaN,NaN,detik,Bencana,"['gempa', 'bumi', 'kuat', 'magnitudo', 'm', 'g..."
1,Gubernur Banten Terpilih Andra Soni Temui Pres...,"Jumat, 13 Des 2024 22:12 WIB",https://news.detik.com/berita/d-7685816/gubern...,Presiden Prabowo Subianto bertemu dengan Guber...,prabowo subianto,andra soni,istana negara,NaN,NaN,detik,Politik,"['presiden', 'prabowo', 'subianto', 'temu', 'g..."
2,"Banjir Besar Landa Jabodetabek, Pimpinan MPR I...",4 Maret 2025,https://nasional.kompas.com/read/2025/03/04/22...,"JAKARTA, KOMPAS.com - Wakil Ketua MPR RI Eddy...",eddy soeparno,mitigasi bencana,banjir jabodetabek,krisis iklim,Banjir Jabodetabek hari ini,kompas,Bencana,"['jakarta', 'kompas', 'com', 'wakil', 'ketua',..."
3,Remaja Ditindak Polisi di Serpong karena Bawa ...,29 April 2025,https://megapolitan.kompas.com/read/2025/04/29...,"TANGERANG SELATAN, KOMPAS.com - Seorang remaj...",polisi,remaja,Mobil berpelat asing,Remaja ditindak polisi,Gunakan mobil pelat asing,kompas,Hukum_Kriminal,"['tangerang', 'selatan', 'kompas', 'com', 'ora..."
4,Tiga Warga Dapat Penanganan Medis Akibat Kebak...,25 Januari 2025,https://megapolitan.kompas.com/read/2025/01/25...,"JAKARTA, KOMPAS.com - Kebakaran yang melanda ...",korban kebakaran,pemadam kebakaran,Kebakaran Jakarta Pusat,kebakaran di mangga besar,kebakaran rumah di mangga besar,kompas,Bencana,"['jakarta', 'kompas', 'com', 'bakar', 'landa',..."


Split Dataset menjadi Train dan Test

In [3]:
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_text'], df['category'], test_size=0.2, random_state=42)

In [4]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [5]:
# Ambil rata-rata skor TF-IDF tiap kata
tfidf_mean = X_train_tfidf.mean(axis=0).A1

words = tfidf.get_feature_names_out()

tfidf_dict = dict(zip(words, tfidf_mean))

top10_tfidf = sorted(tfidf_dict.items(), key=lambda x: x[1], reverse=True)[:10]

print("\nTop 10 Kata Berdasarkan TF-IDF:")
for word, score in top10_tfidf:
    print(f"{word} : {score:.5f}")


Top 10 Kata Berdasarkan TF-IDF:
jakarta : 0.04007
prabowo : 0.03060
kata : 0.02810
jadi : 0.02567
sebut : 0.02422
kpk : 0.02283
presiden : 0.02238
laku : 0.02226
menteri : 0.01862
indonesia : 0.01784


In [6]:
# Tokenisasi
sentences = [text.split() for text in X_train]

w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4
)

In [7]:
all_words = [word for sentence in sentences for word in sentence]
word_freq = Counter(all_words)

top10_words = word_freq.most_common(10)

print("\nTop 10 Kata (Word2Vec - berdasarkan frekuensi):")
for word, freq in top10_words:
    print(word, ":", freq)


Top 10 Kata (Word2Vec - berdasarkan frekuensi):
'jakarta', : 51286
'kata', : 50688
'jadi', : 41513
'sebut', : 40436
'laku', : 29835
'prabowo', : 24798
'presiden', : 20396
'indonesia', : 18955
'jalan', : 18262
'tahun', : 17851


In [8]:
print("\nKata yang mirip (Word2Vec):\n")

for word, _ in top10_words[:5]:  # ambil 5 kata utama
    print(f"Kata: {word}")
    
    try:
        similar_words = w2v_model.wv.most_similar(word, topn=5)
        for sim_word, score in similar_words:
            print(f"  {sim_word} ({score:.3f})")
    except KeyError:
        print("  Tidak ditemukan di vocabulary")
    
    print()


Kata yang mirip (Word2Vec):

Kata: 'jakarta',
  'pramono', (0.461)
  'jakarta'] (0.442)
  'jawa', (0.435)
  'banten', (0.427)
  'job', (0.405)

Kata: 'kata',
  'ujar', (0.929)
  'tutur', (0.895)
  'jelas', (0.815)
  'ucap', (0.810)
  'ungkap', (0.794)

Kata: 'jadi',
  'ini', (0.556)
  'memang', (0.554)
  'bukan', (0.548)
  'suatu', (0.491)
  'arti', (0.487)

Kata: 'sebut',
  'jelas', (0.617)
  'kait', (0.582)
  'jumlah', (0.565)
  'nyata', (0.546)
  'ungkap', (0.535)

Kata: 'laku',
  'proses', (0.548)
  'hasil', (0.534)
  'laksana', (0.492)
  'pihak', (0.484)
  'awal', (0.466)



In [9]:
def document_vector(text):
    words = text.split()
    vec = np.zeros(100)
    count = 0

    for w in words:
        if w in w2v_model.wv:
            vec += w2v_model.wv[w]
            count += 1

    return vec / count if count > 0 else vec

In [10]:
X_train_w2v = np.array([document_vector(text) for text in X_train])
X_test_w2v  = np.array([document_vector(text) for text in X_test])

TF-IDF

In [11]:
joblib.dump(X_train_tfidf, '../models/tfidf_train.pkl')
joblib.dump(X_test_tfidf, '../models/tfidf_test.pkl')
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

['../models/tfidf_vectorizer.pkl']

Word2Vec

In [12]:
X_train_w2v_df = pd.DataFrame(X_train_w2v)
X_train_w2v_df.to_csv('../models/w2v_train.csv', index=False)

X_test_w2v_df = pd.DataFrame(X_test_w2v)
X_test_w2v_df.to_csv('../models/w2v_test.csv', index=False)

print("Word2Vec vector berhasil disimpan!")

Word2Vec vector berhasil disimpan!


In [13]:
w2v_model.save("../models/word2vec.model")

y_train.to_csv('../models/y_train.csv', index=False)
y_test.to_csv('../models/y_test.csv', index=False)

In [17]:
results = []

models_tfidf = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC()
}

print("=== HASIL TF-IDF ===")

for name, model in models_tfidf.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # 🔍 tampil satu per satu
    print(f"\nModel: {name}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")

    results.append({
        "Feature": "TF-IDF",
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    })

=== HASIL TF-IDF ===

Model: Naive Bayes
Accuracy : 0.8348
Precision: 0.8312
Recall   : 0.8348
F1-Score : 0.8319

Model: Logistic Regression
Accuracy : 0.8931
Precision: 0.8920
Recall   : 0.8931
F1-Score : 0.8921

Model: SVM
Accuracy : 0.9150
Precision: 0.9142
Recall   : 0.9150
F1-Score : 0.9137


In [18]:
models_w2v = {
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC()
}

print("\n=== HASIL WORD2VEC ===")

for name, model in models_w2v.items():
    model.fit(X_train_w2v, y_train)
    y_pred = model.predict(X_test_w2v)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # 🔍 tampil satu per satu
    print(f"\nModel: {name}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")

    results.append({
        "Feature": "Word2Vec",
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    })


=== HASIL WORD2VEC ===

Model: Naive Bayes
Accuracy : 0.7763
Precision: 0.7886
Recall   : 0.7763
F1-Score : 0.7809

Model: Logistic Regression
Accuracy : 0.8417
Precision: 0.8397
Recall   : 0.8417
F1-Score : 0.8404

Model: SVM
Accuracy : 0.8733
Precision: 0.8712
Recall   : 0.8733
F1-Score : 0.8705


Perbandingan Performa

In [19]:
df_results = pd.DataFrame(results)

# urutkan dari terbaik
df_results = df_results.sort_values(by="F1-Score", ascending=False)

print("\n=== PERBANDINGAN MODEL ===")
df_results


=== PERBANDINGAN MODEL ===


,Feature,Model,Accuracy,Precision,Recall,F1-Score
2,TF-IDF,SVM,0.914969,0.914202,0.914969,0.913716
1,TF-IDF,Logistic Regression,0.893104,0.892027,0.893104,0.892090
5,Word2Vec,SVM,0.873295,0.871165,0.873295,0.870538
4,Word2Vec,Logistic Regression,0.841712,0.839667,0.841712,0.840392
0,TF-IDF,Naive Bayes,0.834797,0.831227,0.834797,0.831940
3,Word2Vec,Naive Bayes,0.776303,0.788591,0.776303,0.780862
